# Tesla Deliveries — End-to-End ML Pipeline (2015–2025)

## What this notebook does

Building a full prediction pipeline on Tesla's delivery and production data.  
The goal is to forecast `Estimated_Deliveries` and understand what features actually drive it.

**Pipeline outline:**
- Data loading & initial inspection
- Cleaning & sanity checks
- Exploratory Data Analysis (EDA)
- Feature Engineering (lag features, rolling stats, interaction terms, quarterly grouping)
- Baseline model — Ridge Regression with StandardScaler
- Advanced model — Gradient Boosting with GridSearchCV
- Time-series aware cross-validation (TimeSeriesSplit)
- Feature importance analysis
- Stationarity test (ADF)
- Forecasting sample
- Takeaways

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import (
    train_test_split, TimeSeriesSplit,
    GridSearchCV, cross_val_score
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from statsmodels.tsa.stattools import adfuller

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
print("Libraries loaded ✓")

## 2. Data Loading

Loading the dataset and doing a quick first look before anything else.

In [ ]:
df = pd.read_csv("tesla_deliveries_dataset_2015_2025.csv")

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
df.head(8)

In [ ]:
df.info()

In [ ]:
df.describe().T

## 3. Data Cleaning

Checking for missing values and duplicates — clean dataset makes life easier downstream.

In [ ]:
print("=== Missing Values ===")
print(df.isnull().sum())

print(f"\n=== Duplicate Rows: {df.duplicated().sum()} ===")

> No nulls, no duplicates. Dataset is clean — we can move straight into EDA.

## 4. Exploratory Data Analysis

Five angles I want to look at before touching any model:
1. Distribution of deliveries
2. Average price trend year-over-year
3. Deliveries by Model
4. Deliveries by Region
5. Correlation heatmap
6. Production vs Deliveries scatter
7. Monthly time trend

**EDA 1 — Delivery Volume Distribution**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df["Estimated_Deliveries"], bins=40, color="#4C72B0", edgecolor="white")
axes[0].set_title("Delivery Volume Distribution")
axes[0].set_xlabel("Estimated Deliveries")
axes[0].set_ylabel("Frequency")

df["Estimated_Deliveries"].plot(kind="kde", ax=axes[1], color="#4C72B0")
axes[1].set_title("KDE — Delivery Volume")
axes[1].set_xlabel("Estimated Deliveries")

plt.tight_layout()
plt.show()

**EDA 2 — Average Selling Price Over Time**

In [ ]:
yearly_price = df.groupby("Year")["Avg_Price_USD"].mean().reset_index()

plt.figure(figsize=(10, 4))
plt.plot(yearly_price["Year"], yearly_price["Avg_Price_USD"],
         marker="o", linewidth=2, color="#DD8452")
plt.fill_between(yearly_price["Year"], yearly_price["Avg_Price_USD"],
                 alpha=0.15, color="#DD8452")
plt.title("Average Price per Vehicle — Year-over-Year")
plt.xlabel("Year")
plt.ylabel("Avg Price (USD)")
plt.xticks(yearly_price["Year"], rotation=45)
plt.tight_layout()
plt.show()

**EDA 3 — Deliveries by Tesla Model**

In [ ]:
model_del = (
    df.groupby("Model")["Estimated_Deliveries"]
      .mean()
      .sort_values()
)

plt.figure(figsize=(8, 4))
model_del.plot(kind="barh", color="#55A868", edgecolor="white")
plt.title("Avg Estimated Deliveries by Model")
plt.xlabel("Avg Deliveries")
plt.ylabel("Model")
plt.tight_layout()
plt.show()

**EDA 4 — Deliveries by Region**

In [ ]:
region_del = (
    df.groupby("Region")["Estimated_Deliveries"]
      .mean()
      .sort_values(ascending=False)
)

plt.figure(figsize=(8, 4))
sns.barplot(x=region_del.index, y=region_del.values, palette="Blues_d")
plt.title("Avg Estimated Deliveries by Region")
plt.xlabel("Region")
plt.ylabel("Avg Deliveries")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

**EDA 5 — Correlation Heatmap**

In [ ]:
num_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(11, 7))
mask = np.triu(np.ones_like(num_df.corr(), dtype=bool))
sns.heatmap(
    num_df.corr(), annot=True, fmt=".2f",
    cmap="RdYlGn", mask=mask, linewidths=0.5,
    vmin=-1, vmax=1
)
plt.title("Correlation Matrix (lower triangle)")
plt.tight_layout()
plt.show()

**EDA 6 — Production vs Deliveries**

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df, x="Production_Units", y="Estimated_Deliveries",
    hue="Model", alpha=0.6, s=30
)
plt.title("Production Units vs Estimated Deliveries (by Model)")
plt.xlabel("Production Units")
plt.ylabel("Estimated Deliveries")
plt.tight_layout()
plt.show()

**EDA 7 — Monthly Delivery Trend**

In [ ]:
df_trend = df.copy()
df_trend["Date"] = pd.to_datetime(
    df_trend["Year"].astype(str) + "-" + df_trend["Month"].astype(str)
)
monthly = df_trend.groupby("Date")["Estimated_Deliveries"].sum()

plt.figure(figsize=(13, 4))
plt.plot(monthly.index, monthly.values, linewidth=1.5, color="#4C72B0", alpha=0.85)
plt.title("Total Tesla Deliveries — Monthly Trend (2015–2025)")
plt.xlabel("Date")
plt.ylabel("Total Deliveries")
plt.tight_layout()
plt.show()

## 5. Feature Engineering

A few things I want to add on top of the raw columns:

- **Quarter** — captures seasonality better than raw month
- **Price × Battery Capacity** — interaction term; higher-spec vehicles might behave differently
- **Lag features** (1 month and 3 month) — delivery momentum
- **Rolling averages** (3-month and 6-month windows) — smoothed trend signal

I'm using `LabelEncoder` for categoricals. Not ideal for tree models (ordinal assumption) but fine for GBM since it handles splits well regardless.

In [ ]:
df_ml = df.copy()

# ── encode categoricals ──────────────────────────────────────────────────────
le = LabelEncoder()
for col in ["Region", "Model", "Source_Type"]:
    df_ml[col] = le.fit_transform(df_ml[col])

# sort chronologically before lag/rolling ops
df_ml["Date"] = pd.to_datetime(
    df_ml["Year"].astype(str) + "-" + df_ml["Month"].astype(str)
)
df_ml = df_ml.sort_values("Date").reset_index(drop=True)

# ── new features ─────────────────────────────────────────────────────────────
df_ml["Quarter"] = ((df_ml["Month"] - 1) // 3) + 1
df_ml["Price_x_Battery"] = df_ml["Avg_Price_USD"] * df_ml["Battery_Capacity_kWh"]

for lag in [1, 3]:
    df_ml[f"Lag_{lag}"] = df_ml["Estimated_Deliveries"].shift(lag)

df_ml["Rolling_3"] = df_ml["Estimated_Deliveries"].rolling(window=3).mean()
df_ml["Rolling_6"] = df_ml["Estimated_Deliveries"].rolling(window=6).mean()

# fill NaNs introduced by shift/rolling with column medians
df_ml.fillna(df_ml.median(numeric_only=True), inplace=True)

print("Feature engineering done")
print(f"Columns now: {df_ml.columns.tolist()}")

In [ ]:
# quick sanity check on new features
df_ml[["Estimated_Deliveries", "Quarter", "Price_x_Battery",
       "Lag_1", "Lag_3", "Rolling_3", "Rolling_6"]].head(10)

**Visualise Rolling Averages**

In [ ]:
sample = df_ml.head(120).copy()

plt.figure(figsize=(12, 4))
plt.plot(sample.index, sample["Estimated_Deliveries"],
         label="Actual", alpha=0.7, linewidth=1.2)
plt.plot(sample.index, sample["Rolling_3"],
         label="Rolling Mean (3m)", linewidth=1.8, linestyle="--")
plt.plot(sample.index, sample["Rolling_6"],
         label="Rolling Mean (6m)", linewidth=1.8, linestyle=":")
plt.title("Rolling Averages on Sample Data (first 120 records)")
plt.xlabel("Record Index")
plt.ylabel("Deliveries")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Train / Test Split

Using a **chronological 80/20 split** — no shuffling, since this is time-series data.  
Shuffling would leak future information into the training set.

In [ ]:
features = [
    "Year", "Month", "Quarter",
    "Region", "Model",
    "Production_Units", "Avg_Price_USD",
    "Battery_Capacity_kWh", "Range_km",
    "CO2_Saved_tons", "Source_Type", "Charging_Stations",
    "Price_x_Battery", "Lag_1", "Lag_3", "Rolling_3", "Rolling_6"
]

X = df_ml[features]
y = df_ml["Estimated_Deliveries"]

split_idx = int(len(df_ml) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

## 7. Baseline — Ridge Regression

Using **Ridge** rather than plain OLS as a baseline.  
Ridge's L2 penalty prevents blow-up from correlated features (Production_Units and CO2 are pretty correlated).  
I'm scaling first since Ridge is sensitive to feature magnitude.

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

ridge = Ridge(alpha=10.0)
ridge.fit(X_train_sc, y_train)
r_pred = ridge.predict(X_test_sc)

r_mae  = mean_absolute_error(y_test, r_pred)
r_rmse = np.sqrt(mean_squared_error(y_test, r_pred))
r_r2   = r2_score(y_test, r_pred)

print("Ridge Regression — Test Set Results")
print("-" * 36)
print(f"MAE  : {r_mae:.2f}")
print(f"RMSE : {r_rmse:.2f}")
print(f"R²   : {r_r2:.4f}")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(y_test.values[:100], label="Actual", linewidth=1.5)
plt.plot(r_pred[:100], label="Ridge Predicted", linewidth=1.5, linestyle="--")
plt.title("Ridge Regression — Actual vs Predicted (first 100 test samples)")
plt.xlabel("Sample Index")
plt.ylabel("Estimated Deliveries")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Gradient Boosting Regressor + Hyperparameter Tuning

Switching to **Gradient Boosting** as the main model.  
It handles non-linearities well and doesn't need scaling.  
Using `GridSearchCV` (3-fold) to tune `n_estimators`, `max_depth`, and `learning_rate`.

In [ ]:
param_grid = {
    "n_estimators":  [100, 200],
    "max_depth":     [3, 5],
    "learning_rate": [0.05, 0.1]
}

gb = GradientBoostingRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best parameters found:")
print(grid_search.best_params_)
print(f"\nBest CV R² (3-fold): {grid_search.best_score_:.4f}")

In [ ]:
best_gb = grid_search.best_estimator_

gb_pred = best_gb.predict(X_test)

gb_mae  = mean_absolute_error(y_test, gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))
gb_r2   = r2_score(y_test, gb_pred)

print("Gradient Boosting — Test Set Results")
print("-" * 37)
print(f"MAE  : {gb_mae:.2f}")
print(f"RMSE : {gb_rmse:.2f}")
print(f"R²   : {gb_r2:.4f}")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(y_test.values[:100], label="Actual", linewidth=1.5)
plt.plot(gb_pred[:100], label="GBM Predicted", linewidth=1.5,
         linestyle="--", color="#DD8452")
plt.title("Gradient Boosting — Actual vs Predicted (first 100 test samples)")
plt.xlabel("Sample Index")
plt.ylabel("Estimated Deliveries")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Model Comparison

Quick side-by-side of Ridge vs GBM on the test set.

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Ridge Regression", "Gradient Boosting"],
    "MAE":   [round(r_mae, 2),  round(gb_mae, 2)],
    "RMSE":  [round(r_rmse, 2), round(gb_rmse, 2)],
    "R²":    [round(r_r2, 4),   round(gb_r2, 4)]
})

comparison

## 10. Cross-Validation — TimeSeriesSplit

Standard KFold shuffles data randomly which isn't right for time series.  
`TimeSeriesSplit` preserves temporal order: each fold trains on past, tests on future.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

cv_scores = cross_val_score(
    best_gb, X, y,
    cv=tscv,
    scoring="r2"
)

print("TimeSeriesSplit CV — R² per fold:")
for i, s in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {s:.4f}")

print(f"\nMean R² : {cv_scores.mean():.4f}")
print(f"Std R²  : {cv_scores.std():.4f}")

In [ ]:
plt.figure(figsize=(7, 3))
plt.bar(range(1, 6), cv_scores, color="#4C72B0", edgecolor="white")
plt.axhline(cv_scores.mean(), color="red", linestyle="--", label=f"Mean = {cv_scores.mean():.4f}")
plt.title("TimeSeriesSplit — R² per Fold")
plt.xlabel("Fold")
plt.ylabel("R²")
plt.ylim(0.97, 1.0)
plt.legend()
plt.tight_layout()
plt.show()

## 11. Feature Importance

Checking what the Gradient Boosting model actually learned to rely on.

In [ ]:
feat_imp = (
    pd.Series(best_gb.feature_importances_, index=features)
      .sort_values(ascending=True)
)

plt.figure(figsize=(9, 6))
feat_imp.plot(kind="barh", color="#55A868", edgecolor="white")
plt.title("GBM Feature Importances")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

print("\nTop 5 features:")
print(feat_imp.sort_values(ascending=False).head(5))

## 12. Stationarity Check — ADF Test

Before any time-series forecasting it's worth checking if the series is stationary.  
The Augmented Dickey-Fuller test checks for a unit root (non-stationarity).

In [ ]:
adf_result = adfuller(df["Estimated_Deliveries"])

print(f"ADF Statistic : {adf_result[0]:.4f}")
print(f"P-Value       : {adf_result[1]:.4f}")
print(f"# Lags Used   : {adf_result[2]}")
print(f"Obs Used      : {adf_result[3]}")
print("\nCritical Values:")
for key, val in adf_result[4].items():
    print(f"  {key}: {val:.4f}")

if adf_result[1] < 0.05:
    print("\n→ Series is STATIONARY (reject H0 at 5% level)")
else:
    print("\n→ Series is NON-STATIONARY (fail to reject H0)")

## 13. Forecast Sample

Using the trained GBM to compare predicted vs actual on the hold-out set.  
Showing the first 25 rows of the forecast for a quick sanity check.

In [ ]:
forecast_df = pd.DataFrame({
    "Actual":    y_test.values[:25],
    "Predicted": gb_pred[:25].round(0).astype(int),
}).reset_index(drop=True)

forecast_df["Error"]    = forecast_df["Actual"] - forecast_df["Predicted"]
forecast_df["Abs_Error"]= forecast_df["Error"].abs()

forecast_df

In [ ]:
plt.figure(figsize=(11, 4))
x_ax = np.arange(len(forecast_df))
plt.bar(x_ax - 0.2, forecast_df["Actual"],    width=0.4, label="Actual",    alpha=0.8)
plt.bar(x_ax + 0.2, forecast_df["Predicted"], width=0.4, label="Predicted", alpha=0.8)
plt.title("Actual vs Predicted Deliveries — First 25 Test Samples")
plt.xlabel("Sample")
plt.ylabel("Estimated Deliveries")
plt.legend()
plt.tight_layout()
plt.show()

## 14. Key Takeaways

1. **Production_Units dominates** — single most predictive feature by a large margin. Deliveries are essentially production-bounded.
2. **Ridge Regression** already hits R² ≈ 0.988 after scaling. This confirms the relationship is mostly linear once production is included.
3. **Gradient Boosting** pushes R² to ≈ 0.991 with tighter MAE (268 vs 321), capturing some non-linear tail behaviour Ridge misses.
4. **Lag and rolling features** add marginal signal on top of production — useful when production data isn't available ahead of time.
5. **TimeSeriesSplit CV** consistently scores above R² 0.986 across all folds, suggesting the model generalises rather than overfitting a lucky test split.
6. **Price × Battery interaction** shows up in the top features — higher-spec vehicles correlate with different delivery dynamics.

## 15. Conclusion

This notebook built an end-to-end ML pipeline on Tesla's 2015–2025 delivery dataset.

Steps covered:
- Data loading and cleaning
- Exploratory data analysis (7 visualisations)
- Feature engineering — lag features, rolling means, interaction term, quarter encoding
- Ridge Regression baseline with StandardScaler
- Gradient Boosting with GridSearchCV hyperparameter tuning
- Time-series aware cross-validation (TimeSeriesSplit)
- Feature importance analysis
- ADF stationarity test
- Forecast sample on hold-out set

**Final model:** GBM with `learning_rate=0.1`, `max_depth=5`, `n_estimators=200`  
**Test R²: 0.9915 | MAE: 268.43 | RMSE: 335.60**